# 08 - Apresentacao EDA STJ

Notebook de apoio para uma apresentacao de 30 minutos sobre o corpus STJ Integras, recorte 2024-2026.

A ideia e usar os artefatos ja salvos em `data/reports/summaries`, sem reler todos os metadados brutos durante a apresentacao.

## Roteiro sugerido para 30 minutos

1. **Pergunta e fonte dos dados** - 3 min
2. **Escala do corpus** - 5 min
3. **Evolucao temporal** - 5 min
4. **Composicao documental e resultados decisorios** - 6 min
5. **Concentracoes tematicas** - 7 min
6. **Limites atuais e proximos passos** - 4 min

## Escopo e unidade de analise

**Mensagem oral:** esta EDA descreve o corpus de **documentos publicados pelo STJ** entre 2024 e 2026. Ela ainda nao reconstrói a vida processual completa desde a origem em primeira/segunda instancia.

- Unidade observada agora: `SeqDocumento` / documento de integra.
- Chaves auxiliares disponiveis: `processo`, `numeroRegistro` e, quando existir, CNJ normalizado.
- Unidade desejada para a pesquisa longitudinal: processo, com percurso entre instancias, movimentacoes, recursos e decisoes.

Portanto, os numeros desta apresentacao sao bons para entender escala, composicao documental e concentracoes tematicas do corpus STJ. Eles ainda nao devem ser apresentados como uma contagem definitiva de trajetorias processuais completas.

In [3]:
escopo = pd.DataFrame([
    {'camada': 'Documento STJ', 'chave': 'SeqDocumento', 'status': 'capturado nos metadados/ZIPs', 'uso': 'texto, tipo documental, teor, assunto'},
    {'camada': 'Registro STJ', 'chave': 'numeroRegistro', 'status': 'capturado nos metadados', 'uso': 'agrupar quando CNJ nao aparece'},
    {'camada': 'Processo CNJ', 'chave': 'numero_processo', 'status': 'parcial/depende de normalizacao e cruzamentos', 'uso': 'vida processual e origem'},
    {'camada': 'Outras instancias', 'chave': 'CNJ + movimentos', 'status': 'proxima etapa', 'uso': 'trajetoria longitudinal completa'},
])
display(escopo)

,camada,chave,status,uso
0,Documento STJ,SeqDocumento,capturado nos metadados/ZIPs,"texto, tipo documental, teor, assunto"
1,Registro STJ,numeroRegistro,capturado nos metadados,agrupar quando CNJ nao aparece
2,Processo CNJ,numero_processo,parcial/depende de normalizacao e cruzamentos,vida processual e origem
3,Outras instancias,CNJ + movimentos,proxima etapa,trajetoria longitudinal completa


## 1. Ambiente

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

pio.templates.default = 'plotly_white'

COLOR_MAIN = '#19535f'
COLOR_ACCENT = '#0b7a75'
COLOR_WARM = '#7b2d26'
COLOR_GREEN = '#3f6c45'
COLOR_PURPLE = '#5f4b8b'

print('Imports loaded.')

In [4]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'README.md').exists():
            return candidate
        if candidate.name == 'datajud_probe':
            return candidate
    return start

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = find_project_root(Path.cwd())

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if (MOUNT_POINT / 'MyDrive').exists():
        print('Google Drive ja esta montado.')
    else:
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = PROJECT_ROOT / 'data'

REPORTS_DATA = DATA_ROOT / 'reports' / 'summaries'
FIGURES_DATA = DATA_ROOT / 'reports' / 'figures' / 'apresentacao_eda'
FIGURES_DATA.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('REPORTS_DATA:', REPORTS_DATA, 'exists=', REPORTS_DATA.exists())
print('FIGURES_DATA:', FIGURES_DATA)

Google Drive ja esta montado.
DATA_ROOT: /content/drive/MyDrive/Mestrado/2026/llms/data
REPORTS_DATA: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries exists= True
FIGURES_DATA: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/figures/apresentacao_eda


## 2. Carregar tabelas resumidas

In [ ]:
def load_csv(name: str, required: bool = True) -> pd.DataFrame:
    path = REPORTS_DATA / name
    if not path.exists():
        if required:
            raise FileNotFoundError(f'Arquivo nao encontrado: {path}. Rode o notebook 01 antes desta apresentacao.')
        print('Opcional ausente:', path)
        return pd.DataFrame()
    return pd.read_csv(path)


def read_text_if_exists(name: str) -> str:
    path = REPORTS_DATA / name
    return path.read_text(encoding='utf-8') if path.exists() else ''


def parse_summary_number(summary_text: str, label: str) -> int | None:
    pattern = rf'- {re.escape(label)}: ([0-9.,]+)'
    match = re.search(pattern, summary_text)
    if not match:
        return None
    return int(match.group(1).replace('.', '').replace(',', ''))


def compact_number(value: float | int | None) -> str:
    if value is None or pd.isna(value):
        return 'n/a'
    value = float(value)
    if abs(value) >= 1_000_000:
        return f'{value / 1_000_000:.1f} mi'
    if abs(value) >= 1_000:
        return f'{value / 1_000:.0f} mil'
    return f'{value:.0f}'

summary_text = read_text_if_exists('metadata_eda_summary.md')
ciclo_text = read_text_if_exists('stj_ciclo_vida_processual_summary.md')

docs_year = load_csv('stj_docs_by_publication_year.csv')
docs_month = load_csv('stj_docs_by_publication_month.csv')
processes_year = load_csv('stj_processes_by_publication_year.csv', required=False)
docs_type = load_csv('stj_docs_by_type.csv')
docs_minister = load_csv('stj_docs_by_minister.csv')
docs_teor = load_csv('stj_docs_by_teor.csv')
docs_recurso = load_csv('stj_docs_by_recurso.csv')
assuntos_final = load_csv('stj_docs_by_assunto_final_labeled.csv')
assuntos_root = load_csv('stj_docs_by_assunto_root_labeled.csv', required=False)


RECURSO_MAP = {
    "AgInt": "Agravo Interno",
    "AGRAVO INTERNO": "Agravo Interno",
    "EDcl": "Embargos de Declaração",
    "EMBARGOS DE DECLARAÇÃO": "Embargos de Declaração",
    "AgRg": "Agravo Regimental",
    "AGRAVO REGIMENTAL": "Agravo Regimental",
    "RE": "Recurso Extraordinário",
    "RECURSO EXTRAORDINÁRIO": "Recurso Extraordinário",
    "DESIS": "Desistência",
    "PET": "Petição",
    "PETIÇÃO": "Petição",
    "PExt": "Pedido de Extensão",
    "PEDIDO DE EXTENSÃO": "Pedido de Extensão",
    "RCD": "Pedido de Reconsideração",  # validar no texto integral/metadados
    "PEDIDO DE RECONSIDERAÇÃO": "Pedido de Reconsideração",
    "IAC": "Incidente de Assunção de Competência",
}


def normalize_recurso(value: object) -> str:
    text = '(vazio)' if pd.isna(value) else str(value).strip()
    return RECURSO_MAP.get(text, text)


def save_plotly(fig: go.Figure, name: str) -> Path:
    path = FIGURES_DATA / f'{name}.html'
    fig.write_html(path, include_plotlyjs='cdn', full_html=True)
    return path


def show_and_save(fig: go.Figure, name: str) -> go.Figure:
    fig.update_layout(
        font=dict(size=13),
        title_font=dict(size=20),
        margin=dict(l=40, r=30, t=70, b=45),
    )
    save_plotly(fig, name)
    fig.show()
    return fig


tables = {
    'docs_year': docs_year.shape,
    'docs_month': docs_month.shape,
    'processes_year': processes_year.shape,
    'docs_type': docs_type.shape,
    'docs_minister': docs_minister.shape,
    'docs_teor': docs_teor.shape,
    'docs_recurso': docs_recurso.shape,
    'assuntos_final': assuntos_final.shape,
}
tables

## 3. Grandes Numeros

**Mensagem oral:** antes de discutir conteudo, vale mostrar escala e cobertura. O corpus de metadados ja e grande o suficiente para revelar concentracoes institucionais e tematicas, mesmo antes de entrar no texto integral.

In [ ]:
kpis = {
    'Arquivos de origem': parse_summary_number(summary_text, 'Arquivos de origem'),
    'Documentos': parse_summary_number(summary_text, 'Documentos'),
    'Processos unicos': parse_summary_number(summary_text, 'Processos unicos'),
    'Registros unicos': parse_summary_number(summary_text, 'Registros unicos'),
    'SeqDocumento unicos': parse_summary_number(summary_text, 'SeqDocumento unicos'),
}

kpi_df = pd.DataFrame({'metrica': list(kpis.keys()), 'valor': list(kpis.values())})
kpi_df['valor_exibicao'] = kpi_df['valor'].map(compact_number)

fig = make_subplots(
    rows=1,
    cols=len(kpi_df),
    specs=[[{'type': 'indicator'} for _ in range(len(kpi_df))]],
    horizontal_spacing=0.035,
)

card_colors = [COLOR_MAIN, COLOR_ACCENT, COLOR_WARM, COLOR_PURPLE, COLOR_GREEN]
for i, row in kpi_df.iterrows():
    fig.add_trace(
        go.Indicator(
            mode='number',
            value=float(row['valor']) if pd.notna(row['valor']) else 0,
            number={
                'valueformat': ',.0f',
                'font': {'size': 30, 'color': card_colors[i]},
            },
            title={
                'text': f"<b>{row['metrica']}</b><br><span style='font-size:13px'>{row['valor_exibicao']}</span>",
                'font': {'size': 13, 'color': '#222'},
            },
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    title='Escala do corpus STJ Integras - recorte 2024-2026',
    height=360,
    margin=dict(l=24, r=24, t=80, b=24),
    paper_bgcolor='white',
)
show_and_save(fig, '01_kpis_corpus')

kpi_df



## 3.1. Achados derivados para orientar a leitura

**Mensagem oral:** alem dos grandes numeros, alguns indicadores derivados ajudam a transformar a EDA em argumento: ha concentracao por tipo documental, por teor decisorio, por recursos, por relatoria e por temas. Esses achados ainda sao descritivos, mas ja sugerem hipoteses empiricas para a dissertacao.

In [ ]:
total_docs = parse_summary_number(summary_text, 'Documentos') or int(docs_year['documentos'].sum())
processos_unicos = parse_summary_number(summary_text, 'Processos unicos')

# Tipo documental
_type = docs_type.copy()
_type['share'] = _type['documentos'] / total_docs

# Teor
top2_teor = docs_teor.head(2)['documentos'].sum()
share_top2_teor = top2_teor / total_docs

# Recursos
_recurso = docs_recurso.copy()
_recurso['recurso_normalizado'] = _recurso['recurso'].map(normalize_recurso)
recurso_vazio = int(_recurso.loc[_recurso['recurso_normalizado'].eq('(vazio)'), 'documentos'].sum()) if _recurso['recurso_normalizado'].eq('(vazio)').any() else 0
recurso_preenchido = total_docs - recurso_vazio
recurso_norm = (
    _recurso[~_recurso['recurso_normalizado'].eq('(vazio)')]
    .groupby('recurso_normalizado')['documentos']
    .sum()
    .sort_values(ascending=False)
)
share_top3_recursos = recurso_norm.head(3).sum() / recurso_preenchido if recurso_preenchido else np.nan

# Relatoria: tratar como qualidade de dados, nao como inferencia substantiva.
_minister_all = docs_minister.copy()
ministro_vazio = int(_minister_all.loc[_minister_all['ministro'].astype(str).eq('(vazio)'), 'documentos'].sum()) if _minister_all['ministro'].astype(str).eq('(vazio)').any() else 0
share_ministro_vazio = ministro_vazio / total_docs if total_docs else np.nan
_minister = _minister_all[~_minister_all['ministro'].astype(str).eq('(vazio)')]
minister_preenchido = int(_minister['documentos'].sum()) if not _minister.empty else 0
share_top10_ministro_preenchido = _minister.head(10)['documentos'].sum() / minister_preenchido if minister_preenchido else np.nan

# Assuntos
_top_assuntos = assuntos_final.copy()
assunto_total_ocorrencias = _top_assuntos['ocorrencias'].sum()
share_top1_assunto = _top_assuntos.head(1)['ocorrencias'].sum() / assunto_total_ocorrencias if assunto_total_ocorrencias else np.nan
share_top10_assunto = _top_assuntos.head(10)['ocorrencias'].sum() / assunto_total_ocorrencias if assunto_total_ocorrencias else np.nan

# Documento/processo
media_docs_por_processo = total_docs / processos_unicos if processos_unicos else np.nan

achados = pd.DataFrame([
    {'achado': 'Decisoes monocraticas predominam', 'valor': _type.loc[_type['tipoDocumento'].eq('DECISÃO'), 'share'].iloc[0] if _type['tipoDocumento'].eq('DECISÃO').any() else np.nan, 'leitura': 'A base e fortemente documental/decisoria, nao apenas acordaos paradigmaticos.'},
    {'achado': 'Top 2 teores concentram documentos', 'valor': share_top2_teor, 'leitura': 'Nao conhecimento e negativa estruturam boa parte do corpus.'},
    {'achado': 'Recursos preenchidos no corpus', 'valor': recurso_preenchido / total_docs, 'leitura': 'O campo recurso e informativo para uma parcela relevante, mas ha muitos vazios.'},
    {'achado': 'Top 3 recursos entre preenchidos', 'valor': share_top3_recursos, 'leitura': 'AgInt, EDcl e AgRg dominam os recursos informados.'},
    {'achado': 'Campo ministro vazio', 'valor': share_ministro_vazio, 'leitura': 'Este e um alerta metodologico: a relatoria nao deve ser interpretada diretamente a partir desta coluna.'},
    {'achado': 'Top 10 relatores entre preenchidos', 'valor': share_top10_ministro_preenchido, 'leitura': 'Serve apenas para auditoria dos poucos casos preenchidos; precisa de outra fonte para analise substantiva.'},
    {'achado': 'Top assunto final entre ocorrencias', 'valor': share_top1_assunto, 'leitura': 'Trafico de drogas e condutas afins e muito saliente nos codigos.'},
    {'achado': 'Top 10 assuntos finais', 'valor': share_top10_assunto, 'leitura': 'Poucos codigos concentram parte importante das ocorrencias tematicas.'},
    {'achado': 'Media documentos por processo unico', 'valor': media_docs_por_processo, 'leitura': 'Indica proximidade entre documento e processo, mas nao substitui deduplicacao longitudinal.'},
])

achados['valor_exibicao'] = achados['valor'].map(lambda v: f'{v:.1%}' if pd.notna(v) and v <= 1 else f'{v:.2f}' if pd.notna(v) else 'n/a')
display(achados[['achado', 'valor_exibicao', 'leitura']])



In [ ]:
fig = go.Figure(go.Table(
    header=dict(
        values=['Achado descritivo', 'Valor', 'Leitura para a dissertação'],
        fill_color=COLOR_MAIN,
        font=dict(color='white', size=13),
        align='left',
    ),
    cells=dict(
        values=[achados['achado'], achados['valor_exibicao'], achados['leitura']],
        fill_color='#f7f9fa',
        align='left',
        height=32,
    )
))
fig.update_layout(title='Achados derivados: da EDA para hipóteses de pesquisa', height=560)
show_and_save(fig, '01b_achados_derivados')

## 4. Evolucao temporal

**Mensagem oral:** 2025 e o ano mais volumoso do recorte completo. 2026 ainda esta parcial, entao qualquer comparacao anual precisa ser feita com cuidado.

In [ ]:
plot_year = docs_year.copy()
plot_year = plot_year[~plot_year['ano_dataPublicacao'].astype(str).eq('(vazio)')].copy()
plot_year['ano_dataPublicacao'] = plot_year['ano_dataPublicacao'].astype(float).astype(int).astype(str)
plot_year['documentos_label'] = plot_year['documentos'].map(compact_number)

fig = px.bar(
    plot_year,
    x='ano_dataPublicacao',
    y='documentos',
    text='documentos_label',
    color_discrete_sequence=[COLOR_ACCENT],
    title='Documentos por ano de publicação',
    labels={'ano_dataPublicacao': 'Ano de publicação', 'documentos': 'Documentos'},
)
fig.update_traces(textposition='outside', hovertemplate='Ano %{x}<br>Documentos: %{y:,}<extra></extra>')
fig.update_yaxes(tickformat=',.0f')
fig.update_layout(height=520, uniformtext_minsize=10, uniformtext_mode='hide')
show_and_save(fig, '02_documentos_por_ano')

plot_year

In [ ]:
month = docs_month.copy()
month = month[~month['mes_dataPublicacao'].astype(str).isin(['NaT', '(vazio)'])].copy()
month['mes_dataPublicacao'] = pd.PeriodIndex(month['mes_dataPublicacao'], freq='M').to_timestamp()
month = month.sort_values('mes_dataPublicacao')

fig = px.line(
    month,
    x='mes_dataPublicacao',
    y='documentos',
    markers=True,
    title='Documentos por mês de publicação',
    labels={'mes_dataPublicacao': 'Mês', 'documentos': 'Documentos'},
)
fig.update_traces(line=dict(color=COLOR_MAIN, width=3), marker=dict(size=8), hovertemplate='%{x|%Y-%m}<br>Documentos: %{y:,}<extra></extra>')
fig.update_yaxes(tickformat=',.0f')
fig.update_layout(height=540)
show_and_save(fig, '03_documentos_por_mes')

month.tail()

## 5. Documentos, teor e recursos

**Mensagem oral:** a base mistura decisoes monocraticas e acordaos. Em termos de teor, predominam documentos que negam ou nao conhecem recursos/pedidos, o que e crucial para pensar a funcao institucional do STJ como tribunal de precedentes e filtro recursal.

No grafico de recursos, siglas como `AgInt`, `EDcl`, `AgRg` e `RE` foram normalizadas para descricoes legiveis. `DESIS` foi mapeado como `Desistência`, mas deve ser validado em amostra documental olhando `descricaoMonocratica`, `teor` e texto integral.

In [ ]:
type_plot = docs_type.head(10).copy()
teor_plot = docs_teor.head(8).iloc[::-1].copy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Tipo de documento', 'Teor mais frequente'),
    horizontal_spacing=0.18,
)
fig.add_trace(
    go.Bar(x=type_plot['tipoDocumento'], y=type_plot['documentos'], marker_color=COLOR_WARM, text=type_plot['documentos'].map(compact_number), textposition='outside', name='Tipo'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=teor_plot['documentos'], y=teor_plot['teor'], orientation='h', marker_color=COLOR_GREEN, text=teor_plot['documentos'].map(compact_number), textposition='outside', name='Teor'),
    row=1,
    col=2,
)
fig.update_xaxes(title_text='Tipo documental', row=1, col=1)
fig.update_yaxes(title_text='Documentos', tickformat=',.0f', row=1, col=1)
fig.update_xaxes(title_text='Documentos', tickformat=',.0f', row=1, col=2)
fig.update_layout(title='Composição documental e teor decisório', height=540, showlegend=False)
show_and_save(fig, '04_tipo_e_teor')

In [ ]:
recurso_plot = docs_recurso.copy()
recurso_plot['recurso_normalizado'] = recurso_plot['recurso'].map(normalize_recurso)
recurso_norm = (
    recurso_plot
    .groupby('recurso_normalizado', dropna=False)['documentos']
    .sum()
    .reset_index()
    .sort_values('documentos', ascending=False)
)
recurso_norm = recurso_norm[~recurso_norm['recurso_normalizado'].astype(str).eq('(vazio)')]
recurso_norm['documentos_label'] = recurso_norm['documentos'].map(compact_number)
recurso_top = recurso_norm.head(15).iloc[::-1]

fig = px.bar(
    recurso_top,
    x='documentos',
    y='recurso_normalizado',
    orientation='h',
    text='documentos_label',
    color_discrete_sequence=[COLOR_PURPLE],
    title='Recursos mais frequentes, com siglas normalizadas',
    labels={'documentos': 'Documentos', 'recurso_normalizado': 'Recurso'},
)
fig.update_traces(textposition='outside', hovertemplate='%{y}<br>Documentos: %{x:,}<extra></extra>')
fig.update_xaxes(tickformat=',.0f')
fig.update_layout(height=620)
show_and_save(fig, '05_recursos')

print('Para validar DESIS no notebook 01, use:')
print('metadata_eda.loc[metadata_eda["recurso"].eq("DESIS"), ["SeqDocumento", "processo", "tipoDocumento", "recurso", "teor", "descricaoMonocratica"]].head(20)')
recurso_norm.head(12)

### Leitura dos recursos

**Mensagem oral:** a normalizacao das siglas ajuda a explicar o que esta sendo contado. O dominio de Agravo Interno, Embargos de Declaracao e Agravo Regimental reforca a leitura de um corpus muito associado a dinamicas recursais e de filtro.

In [ ]:
recurso_dict_table = pd.DataFrame(
    sorted(RECURSO_MAP.items()),
    columns=['sigla_ou_nome_original', 'descricao_normalizada'],
)
display(recurso_dict_table)

fig = go.Figure(go.Table(
    header=dict(values=['Sigla/nome original', 'Descrição normalizada'], fill_color=COLOR_PURPLE, font=dict(color='white'), align='left'),
    cells=dict(values=[recurso_dict_table['sigla_ou_nome_original'], recurso_dict_table['descricao_normalizada']], fill_color='#f7f5fb', align='left', height=28),
))
fig.update_layout(title='Dicionário usado para normalizar recursos', height=560)
show_and_save(fig, '05b_dicionario_recursos')

## 6. Relatoria nos metadados

**Mensagem oral:** depois de combinar `ministro`, `NM_MINISTRO` e `relator`, o campo passa a ser informativo para concentração documental por relator no STJ. Ainda assim, ele descreve documentos do corpus STJ, não a vida completa do processo em todas as instâncias.

In [ ]:
minister = docs_minister.copy()
minister['is_vazio'] = minister['ministro'].astype(str).eq('(vazio)')
empty_minister = int(minister.loc[minister['is_vazio'], 'documentos'].sum()) if minister['is_vazio'].any() else 0
non_empty = minister.loc[~minister['is_vazio']].head(12).iloc[::-1].copy()

total_docs = int(docs_year['documentos'].sum())
empty_pct = empty_minister / total_docs if total_docs else np.nan
non_empty['documentos_label'] = non_empty['documentos'].map(compact_number)

title = 'Relatoria nos metadados STJ'
if empty_minister:
    title += f' (vazio = {empty_pct:.1%})'

fig = px.bar(
    non_empty,
    x='documentos',
    y='ministro',
    orientation='h',
    text='documentos_label',
    color_discrete_sequence=[COLOR_ACCENT],
    title=title,
    labels={'documentos': 'Documentos', 'ministro': 'Relator/ministro'},
)
fig.update_traces(textposition='outside', hovertemplate='%{y}<br>Documentos: %{x:,}<extra></extra>')
fig.update_xaxes(tickformat=',.0f')
fig.update_layout(height=620)
show_and_save(fig, '06_ministros_preenchidos')

minister.head(10)

## 7. Concentracao tematica

**Mensagem oral:** os assuntos mostram concentracao penal, mas os nomes estao sendo usados como rotulos auxiliares dos codigos. Neste momento, a fonte local do lookup e a tabela da Justica Federal de 1º Grau; portanto, nao devo apresentar a coluna `instancia` como cobertura real de todas as instancias. Para a apresentacao, o ponto forte e a concentracao dos **codigos de assunto**; o enriquecimento textual e provisório e precisa ser ampliado com tabelas de outras instancias/CNJ.

In [ ]:
assuntos_plot = assuntos_final.copy()
label_col = 'assunto' if 'assunto' in assuntos_plot.columns else 'assunto_final'
assuntos_plot[label_col] = assuntos_plot[label_col].fillna('(sem rotulo)')
assuntos_plot['rotulo'] = assuntos_plot['assunto_final'].astype(str).str.zfill(5) + ' - ' + assuntos_plot[label_col].astype(str)
assuntos_plot = assuntos_plot.head(15).iloc[::-1].copy()
assuntos_plot['ocorrencias_label'] = assuntos_plot['ocorrencias'].map(compact_number)

fig = px.bar(
    assuntos_plot,
    x='ocorrencias',
    y='rotulo',
    orientation='h',
    text='ocorrencias_label',
    color_discrete_sequence=[COLOR_WARM],
    title='Códigos finais de assunto mais frequentes',
    labels={'ocorrencias': 'Ocorrências', 'rotulo': 'Assunto'},
)
fig.update_traces(textposition='outside', hovertemplate='%{y}<br>Ocorrências: %{x:,}<extra></extra>')
fig.update_xaxes(tickformat=',.0f')
fig.update_layout(height=760)
show_and_save(fig, '07_top_assuntos')

assuntos_plot[['assunto_final', 'assunto', 'ocorrencias', 'caminho_assuntos', 'rotulo']].tail()

In [ ]:
if not assuntos_root.empty and {'assunto_raiz', 'ocorrencias'}.issubset(assuntos_root.columns):
    root_plot = assuntos_root.copy()
    label_col = 'assunto' if 'assunto' in root_plot.columns else 'assunto_raiz'
    root_plot[label_col] = root_plot[label_col].fillna('(sem rotulo)')
    root_plot['rotulo'] = root_plot['assunto_raiz'].astype(str).str.zfill(5) + ' - ' + root_plot[label_col].astype(str)
    root_plot = root_plot.head(12).iloc[::-1].copy()
    root_plot['ocorrencias_label'] = root_plot['ocorrencias'].map(compact_number)

    fig = px.bar(
        root_plot,
        x='ocorrencias',
        y='rotulo',
        orientation='h',
        text='ocorrencias_label',
        color_discrete_sequence=[COLOR_MAIN],
        title='Códigos raiz de assunto mais frequentes',
        labels={'ocorrencias': 'Ocorrências', 'rotulo': 'Assunto raiz'},
    )
    fig.update_traces(textposition='outside', hovertemplate='%{y}<br>Ocorrências: %{x:,}<extra></extra>')
    fig.update_xaxes(tickformat=',.0f')
    fig.update_layout(height=700)
    show_and_save(fig, '08_assuntos_raiz')
else:
    print('Tabela de assuntos raiz nao disponivel.')

## 8. Limite atual: vida processual completa

**Mensagem oral:** esta parte nao deve ser vendida como resultado final. Ela mostra que a camada documental esta forte, mas a espinha dorsal processo-CNJ ainda precisa ser consolidada. A vida processual completa exigira juntar `numeroRegistro`, CNJ, Atas de Distribuicao, Movimentacao Processual, DataJud e possivelmente dados de outras instancias.

In [ ]:
def parse_ciclo_metric(label: str) -> int | None:
    return parse_summary_number(ciclo_text, label)

ciclo_metrics = {
    'Processos na espinha dorsal': parse_ciclo_metric('Processos na espinha dorsal'),
    'Eventos de distribuicao/registro na ata': parse_ciclo_metric('Eventos de distribuicao/registro na ata'),
    'Movimentos DataJud': parse_ciclo_metric('Movimentos DataJud'),
    'Documentos no manifesto': parse_ciclo_metric('Documentos de integras no manifesto'),
    'Chaves auxiliares no corpus': parse_ciclo_metric('Chaves auxiliares no corpus textual'),
}

metrics_df = pd.DataFrame({'metrica': list(ciclo_metrics.keys()), 'valor': list(ciclo_metrics.values())})
metrics_df['valor_exibicao'] = metrics_df['valor'].map(compact_number)

fig = go.Figure()
fig.add_trace(go.Table(
    header=dict(values=['Métrica', 'Valor'], fill_color=COLOR_MAIN, font=dict(color='white', size=14), align='left'),
    cells=dict(values=[metrics_df['metrica'], metrics_df['valor_exibicao']], fill_color='#f5f7f8', align='left', height=30),
    domain=dict(x=[0.0, 0.47], y=[0.0, 0.82]),
))
fig.add_annotation(
    x=0.55,
    y=0.72,
    xref='paper',
    yref='paper',
    text='Leitura: já há um corpus documental amplo,<br>mas a chave processo-CNJ ainda não está resolvida<br>para uma espinha dorsal longitudinal.',
    showarrow=False,
    align='left',
    font=dict(size=15, color='#222'),
)
fig.add_annotation(
    x=0.55,
    y=0.36,
    xref='paper',
    yref='paper',
    text='Próximo passo: conectar documento, registro STJ,<br>CNJ, atas, movimentações e outras instâncias.',
    showarrow=False,
    align='left',
    font=dict(size=15, color='#222'),
)
fig.update_layout(title='Limite metodológico: ligação processual ainda incompleta', height=480)
show_and_save(fig, '09_limite_ciclo_vida')

ciclo_metrics

## 9. Fechamento para a apresentacao

**Tres conclusoes defensaveis agora:**

1. O corpus 2024-2026 e grande e operacionalizavel: mais de um milhao de documentos e processos unicos nos metadados.
2. A base tem concentracoes claras: decisoes predominam, o teor e fortemente concentrado em `Nao Conhecendo` e `Negando`, e assuntos penais aparecem no topo.
3. A pesquisa agora tem dois caminhos tecnicos bem definidos: baixar/processar os ZIPs de texto em lote e resolver a ligacao longitudinal processo-CNJ-registro STJ, eventualmente incorporando dados de outras instancias.

**Frase de transicao para o professor:**

> A EDA sugere que a base STJ e viavel para pesquisa empirica com LLMs, mas a etapa critica nao e apenas textual: e construir uma unidade processual confiavel, capaz de conectar documento, registro STJ, CNJ e, quando possivel, dados de outras instancias.

## 10. Figuras exportadas

In [ ]:
exported = sorted(FIGURES_DATA.glob('*.html'))
print('Figuras Plotly exportadas:', len(exported))
for path in exported:
    print('-', path)